# Quickstart

`sprotmetrics` scores the preprocessing choices behind a multiplexed imaging
experiment: how the image was corrected, how cells were segmented, how pixels
were aggregated into a cell table, what was filtered, how it was normalized.

This page computes metrics on a small synthetic cell table so you can see the
shape of the output. [Comparing two segmentations](comparing-segmentations)
puts them to work on an actual decision.

## Some data to score

Any `AnnData` with cells as rows and markers as columns will do. Metrics that
need more than the matrix say so: cell area lives in `obs["area"]`, positions in
`obs["centroid_y"]` / `obs["centroid_x"]`, and QC status in `obs["passed_QC"]`.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
markers = ["DNA_1", "DNA_2", "CD45", "ECAD"]
n_cells = 300

X = rng.lognormal(3.0, 1.0, size=(n_cells, len(markers))).astype(np.float32)
# DNA_2 is the same nuclei imaged again in a later cycle, so it tracks DNA_1.
X[:, 1] = np.abs(X[:, 0] * rng.normal(1.0, 0.05, n_cells)).astype(np.float32)

# A quarter of the objects are debris: small and dim, which is what QC is for.
debris = np.zeros(n_cells, bool)
debris[: n_cells // 4] = True
X[debris] *= 0.05

adata = ad.AnnData(
    X=X.copy(),
    obs=pd.DataFrame(
        {
            "area": np.where(debris, rng.uniform(5, 40, n_cells), rng.uniform(80, 300, n_cells)),
            "centroid_y": rng.uniform(0, 500, n_cells),
            "centroid_x": rng.uniform(0, 500, n_cells),
            "passed_QC": ~debris,
        },
        index=[str(i + 1) for i in range(n_cells)],
    ),
)
adata.var_names = markers
adata.layers["counts"] = X.copy()
adata.layers["normalized"] = np.arcsinh(X / 5.0).astype(np.float32)
adata

AnnData object with n_obs × n_vars = 300 × 4
    obs: 'area', 'centroid_y', 'centroid_x', 'passed_QC'
    layers: None (.X), 'counts', 'normalized'

## Computing a family

The quickest way in is a *family*: one call returning every metric that scores a
given aspect of the run.

In [2]:
import sprotmetrics as spm

spm.basic_statistics(adata)

{'n_cells': 300.0,
 'n_markers': 4.0,
 'qc_pass_rate': 0.75,
 'median_area': 141.25250172030908,
 'mean_area': 141.71514605265796,
 'median_total_intensity': 73.05722498893738,
 'mean_total_intensity': 95.35653123039752,
 'median_marker_coverage': 0.9933333333333333,
 'min_marker_coverage': 0.9933333333333333}

## QC-passing variants

QC steps in imaging pipelines usually *annotate* rather than filter, so the
all-cell metrics come out the same whichever QC method ran. Asking for
`qc_variants=True` reports each metric a second time over the cells that passed,
under a `_qc_pass` suffix — and that is where the choice of QC method actually
becomes visible.

In [3]:
scores = spm.basic_statistics(adata, qc_variants=True)

pd.DataFrame(
    {
        "all cells": {k: v for k, v in scores.items() if not k.endswith("_qc_pass")},
        "QC-passing": {k[: -len("_qc_pass")]: v for k, v in scores.items() if k.endswith("_qc_pass")},
    }
).dropna()

,all cells,QC-passing
n_cells,300.000000,225.000000
median_area,141.252502,177.598292
mean_area,141.715146,181.356789
median_total_intensity,73.057225,101.553077
mean_total_intensity,95.356531,124.966672
median_marker_coverage,0.993333,1.000000
min_marker_coverage,0.993333,1.000000


`median_area` more than doubles once the debris is excluded, which is exactly
what you would hope: the filter removed the small objects.

Two metrics have no `_qc_pass` twin, and that is deliberate rather than an
oversight. `n_markers` cannot change, and `qc_pass_rate` is 1 by definition
among the cells that passed.

## Looking underneath a summary

Family functions return panel-level numbers. The useful follow-up question is
usually not *what is the median* but *which marker is dragging it down*, so the
per-marker quantities are public too.

In [4]:
spm.marker_table(adata)[["mean", "cv", "coverage", "dynamic_range"]]

,mean,cv,coverage,dynamic_range
DNA_1,23.911090,1.321413,0.993333,138.022497
DNA_2,24.020507,1.322735,0.993333,146.614558
CD45,22.286653,1.440873,0.993333,124.818145
ECAD,25.138281,1.510864,0.996667,187.056350


In [5]:
# The same quantity the family reduces, one value per marker.
pd.Series(spm.marker_coverage(adata), index=adata.var_names, name="coverage")

DNA_1    0.993333
DNA_2    0.993333
CD45     0.993333
ECAD     0.996667
Name: coverage, dtype: float64

## The other families

Each covers one aspect of the run, and each takes `qc_variants` and `subset`.

| Family | Scores |
| --- | --- |
| `basic_statistics` | object counts, areas, intensities |
| `marker_distribution` | per-marker brightness, spread, detection rate |
| `normalization_diagnostics` | what the normalization did to the data |
| `covariate_dependence` | residual dependence on cell area |
| `segmentation_quality` | whether objects look like single cells |
| `signal_background` | pixel-level signal against local background |
| `cluster_separability` | whether cell types stay separable |

Some need configuration, and when it is missing they return `NaN` instead of
guessing. `segmentation_quality` cannot know which of your channels are nuclear
or which marker pairs are biologically incompatible, so you tell it.

In [6]:
spm.segmentation_quality(
    adata,
    nuclear_markers=["DNA_1", "DNA_2"],
    exclusive_pairs=[("CD45", "ECAD")],
)

{'dapi_positive_ratio': 0.59,
 'replicate_correlation': 0.9975246987239649,
 'median_replicate_cv': 0.01833875889223059,
 'mutually_exclusive_coexpression_ratio': 0.7017543859649122,
 'spillover_ratio': 0.5182455553675664}

In [7]:
# Without the panel configuration, the metrics that need it decline to answer.
spm.segmentation_quality(adata)

{'dapi_positive_ratio': 0.59,
 'replicate_correlation': 0.9975246987239649,
 'median_replicate_cv': 0.01833875889223059,
 'mutually_exclusive_coexpression_ratio': nan,
 'spillover_ratio': 0.5182455553675664}

## Next

- [Comparing two segmentations](comparing-segmentations) — using these metrics
  to choose between two pipeline runs, and why no single number settles it.
- [Choosing metrics](choosing-metrics) — which metrics can rank a method at all.